# Assignment 3: train Model 1 / Model 3 on Colab

Runs our repo code (`src/`) on the full KITTI dataset. Run the cells top to bottom.

| Cell | What it does |
| :--- | :--- |
| 1 | Settings (model, epochs, batch size, resume) + mount Drive |
| 2 | Copy the dataset from Drive to the fast Colab disk (`/content/kitti`) |
| 3 | Our code, defined right here (same code as `assignment3/src/`; only Model 1 and Model 3 registered) |
| 4 | Config (`configs/base.yaml` as a dict, only the keys the code reads) |
| 5 | Setup: fresh start or resume (both on Google Drive) |
| 6 | Train: saves `last.pt`, `best.pt`, `log.csv` to Google Drive every epoch |
| 7 | Show the log (can run any time after an epoch finishes) |
| 8 | Close: finish the upload to Drive + shut down the runtime |

**Everything is on Google Drive:** `MyDrive/KITTI_DC/checkpoints/m1/` or `.../m3/`.

- **Fresh** (`RESUME` unchecked): starts at epoch 0. An old run in that folder is kept, renamed to `m1_old_<date-time>`.
- **Resume** (`RESUME` checked): continues from `last.pt` on Drive. Keep `EPOCHS` the same as the first run.
- If the runtime disconnects, at most the current epoch is lost: tick `RESUME` and run all cells again.

### A100 high-RAM version (Runtime > Change runtime type > A100 + High-RAM)
Faster than `training.ipynb`, same model / loss / metrics:
- **`CACHE_IN_RAM = True`**: cell 6 first loads + transforms all train and val samples once (all CPU cores, a few minutes),
  keeps them in RAM (~95 GB of ~167 GB), then every epoch reads straight from RAM (no PNG decoding, no transform).
- LiDAR mask computed with numpy instead of a per-pixel Python loop (same values).
- `cudnn.benchmark` on (fixed 352x1216 input).
- `BATCH_SIZE = 16` by default: use the **same batch size for m1 and m3**.
- The cache is lost when the runtime ends: a resumed session rebuilds it first.

In [1]:
# @title 1. Settings + mount Drive
MODEL = "m3_split"       # @param ["m1_rgb", "m3_early", "m3_split"]
EPOCHS = 20              # @param {type:"integer"}
BATCH_SIZE = 8          # @param {type:"integer"}
MAX_BATCHES = 0          # @param {type:"integer"}
RESUME = False           # @param {type:"boolean"}
CACHE_IN_RAM = False      # @param {type:"boolean"}
# MAX_BATCHES > 0 = quick smoke run (only that many batches per epoch)
# RESUME = True  -> continue from last.pt saved on Drive; False -> start from epoch 0
# CACHE_IN_RAM = True -> transform all train + val samples once and keep them in RAM (~95 GB)
#   only on a runtime with >= 120 GB System RAM (A100 / H100 high-RAM); L4 (53 GB) -> keep False

DATA_DIR = "/content/kitti"                                     # dataset on the fast Colab disk
DRIVE_TARS = "/content/drive/MyDrive/KITTI_DC/dataset"          # train.tar / val.tar / test.tar
DRIVE_RUNS = "/content/drive/MyDrive/KITTI_DC/checkpoints"

import os, torch
from google.colab import drive
drive.mount("/content/drive")

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi
!free -g

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
Sat Nov 28 17:27:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             56W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabl

In [2]:
# @title 2. Load the dataset to the Colab disk (skipped if already there)
import subprocess

if not all(os.path.isdir(f"{DATA_DIR}/{s}") for s in ("train", "val", "test")):
    os.makedirs(DATA_DIR, exist_ok=True)
    for s in ("train", "val", "test"):
        print("extracting", s)
        subprocess.run(["tar", "-xf", f"{DRIVE_TARS}/{s}.tar", "-C", DATA_DIR], check=True)

for s in ("train", "val", "test"):
    print(s, len(os.listdir(f"{DATA_DIR}/{s}/rgb")), "pairs")

extracting train
extracting val
extracting test
train 8237 pairs
val 1023 pairs
test 1000 pairs


## 3. Our code
The same code as the repo files in `assignment3/src/`, defined right here (nothing is saved to disk).
Run these cells in order. After editing one, run it again, then run cells 5 and 6 again.

In [3]:
# @title src/cost.py
import time

import torch

class TrainingTimer:
    def __init__(self, device):
        self.device = device
        self.time = 0
        self.device_name = torch.cuda.get_device_name(self.device) if self.device.type =="cuda" else "cpu"

    def __enter__(self):
        if self.device.type == "cuda": torch.cuda.synchronize(self.device)
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.device.type == "cuda": torch.cuda.synchronize(self.device)
        self._end = time.perf_counter()
        self.time = self._end - self._start

    def export(self):
        return {"device": self.device, "time": self.time}

In [4]:
# @title src/data_and_transform.py
"""Transform works on 1 sample; get_data stacks samples into batches."""
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from torchvision.models import ResNet34_Weights

# ImageNet MEAN / STD for the pretrained encoder, shape (3, 1, 1) to fit a (3, H, W) rgb
_preset = ResNet34_Weights.IMAGENET1K_V1.transforms()
MEAN = np.array(_preset.mean, np.float32)[:, None, None]
STD = np.array(_preset.std, np.float32)[:, None, None]
MAX_DEPTH = 80.0   # team contract: lidar channel 0 = depth / 80


class Transform:
    """Decoded arrays of one sample -> tensors (C, H, W)."""

    def rgb_to_tensor(self, rgb):
        """(3, H, W) 0..1 -> (3, H, W) standardised."""
        rgb = (rgb - MEAN) / STD
        return torch.from_numpy(rgb.copy())

    def lidar_to_tensor(self, lid):
        """(H, W) metres -> (2, H, W): [depth / MAX_DEPTH, mask]."""
        depth = lid / MAX_DEPTH
        mask = (lid > 0).astype(np.float32)    # same values as the per-pixel loop, ~80x faster
        return torch.from_numpy(np.stack([depth, mask]).copy())

    def gt_to_tensor(self, gt):
        """(H, W) metres -> (1, H, W) metres."""
        return torch.from_numpy(gt[None].copy())

    def __call__(self, rgb, lid, gt):
        """Returns {'rgb': (3, H, W), 'lidar': (2, H, W), 'gt': (1, H, W)}."""
        return {"rgb": self.rgb_to_tensor(rgb), "lidar": self.lidar_to_tensor(lid), "gt": self.gt_to_tensor(gt)}


class KittiDepthDataset:
    """Split folder input_dir/{rgb,lidar,gt}/<name>.png -> batches."""

    def __init__(self,input_dir, output_dir, split, transform, batch_size=10, seed=666, cache=False):
        """Store settings and list file names (nothing loaded)."""
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.split = split
        self.transform = transform
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.names = sorted(p.name for p in (self.input_dir/"rgb").glob("*.png"))
        self.cache = None
        if cache:    # load + transform every sample once (all CPU cores), keep the tensors in RAM
            self.cache = []
            with ThreadPoolExecutor(os.cpu_count()) as ex:
                for i, s in enumerate(ex.map(lambda n: self.process(*self.load(n)), self.names)):
                    self.cache.append(s)
                    if (i + 1) % 1000 == 0 or i + 1 == len(self.names):
                        print(f"  cached {split}: {i+1}/{len(self.names)}")

    def load(self, name):
        """Read 3 PNGs, raw values: rgb (H, W, 3) 0..255, lid / gt (H, W) metres * 256."""
        rgb = np.asarray(Image.open(self.input_dir/"rgb"/name).convert('RGB'), dtype=np.float32)
        lid = np.asarray(Image.open(self.input_dir/"lidar"/name), dtype=np.float32)
        gt = np.asarray(Image.open(self.input_dir/"gt"/name), dtype=np.float32)
        return rgb, lid, gt

    def process(self, rgb, lid, gt):
        """Decode to rgb 0..1 / metres, then transform."""
        rgb = rgb.astype(np.float32).transpose(2,0,1)/255.0
        lid = lid.astype(np.float32)/256.0
        gt = gt.astype(np.float32)/256.0
        return self.transform(rgb, lid, gt)

    def __len__(self):
        return len(self.names)

    def get_batch(self):
        """Yield batches until all samples are used (train is shuffled)."""
        order = np.arange(len(self.names))
        if self.split == "train": self.rng.shuffle(order)
        for start in range(0, len(order), self.batch_size):
            end = len(order) if start+self.batch_size > len(order) else start+self.batch_size
            batch_idx = order[start:end]
            samples = []
            for internal_idx in batch_idx:
                if self.cache is not None:    # already transformed, from RAM
                    samples.append(self.cache[internal_idx])
                    continue
                rgb, lid, gt = self.load(self.names[internal_idx])
                samples.append(self.process(rgb, lid, gt))
            batch = {
                "rgb" : torch.stack([s["rgb"] for s in samples]),
                "lidar" : torch.stack([s["lidar"] for s in samples]),
                "gt" : torch.stack([s["gt"] for s in samples])
            }
            yield batch

In [5]:
# @title src/loss_and_metric.py
import torch

def  _valid(pred, gt):
    mask = gt > 0
    p = torch.masked_select(pred,  mask)
    g = torch.masked_select(gt,  mask)
    #1D array  as output
    return p, g
###Loss  function
#rgb: mae
def m1_loss(pred,  gt):
    p, g = _valid(pred,  gt)
    return ((p-g)**2).mean().sqrt()

#depth: rmse
def m2_loss(pred,  gt):
    p, g = _valid(pred,  gt)
    return ((p-g)**2).mean().sqrt()

#early fusion: w1*mae + w2*rmse
def m3_loss(pred, gt,w1=1,w2=1):
    p, g = _valid(pred,  gt)
    return ((p-g)**2).mean().sqrt()

#custom fusion: #TODO Thanh does it bro

###Metric:
class Metric:
    def __init__(self, max_depth=80, by_range=True):
        self.max_depth = max_depth
        self.by_range = by_range
        self.ranges = [(0, 20), (20, 40), (40, max_depth)]
        self.reset()

    def reset(self):
        self.sum_sq = 0
        self.sum_abs = 0
        self.total_valid  = 0
        self.total_pixels = 0
        self.ranged_metric = {r:[0.0,0.0,0] for r in self.ranges}

    @torch.no_grad()
    def update(self, pred, gt):
        pred = pred.float().clamp(0, self.max_depth)
        p, g = _valid(pred, gt.float())
        e = p-g

        self.sum_sq += (e**2).sum().item()
        self.sum_abs += (e).abs().sum().item()
        self.total_valid  += e.numel()
        self.total_pixels += pred.numel()

        if  self.by_range:
            for low,  high in self.ranges:
                in_range  = (g >= low) & ((g <= high) if high == self.max_depth else (g < high))
                range = torch.masked_select(e, in_range)
                self.ranged_metric[(low, high)][0] += (range**2).sum().item()
                self.ranged_metric[(low, high)][1] += (range).abs().sum().item()
                self.ranged_metric[(low, high)][2] += range.numel()

    def compute(self):
        if self.total_valid  == 0: raise ValueError("no valid pixels (gt > 0) - check the data")
        n = self.total_valid
        out = {
            "rmse": (self.sum_sq / n) ** 0.5,                 # metres
            "mae": self.sum_abs / n,                          # metres
            "total_valid ": self.total_valid ,
            "%valid/total_pixel": self.total_valid/self.total_pixels
        }
        if self.by_range:
            for (lo, hi), (sq, ab, n_r) in self.ranged_metric.items():
                name = f"{lo}-{hi}"
                out[name] = {
                    "rmse": (sq / n_r) ** 0.5 if n_r else float("nan"),
                    "mae": ab / n_r if n_r else float("nan"),
                    "total_valid ": n_r,
                    "%valid/total_valid": n_r / self.total_valid,
                }
        return out

In [6]:
# @title src/models/encoder.py
from torchvision.models import resnet34, ResNet34_Weights
import torch.nn as nn
import torch

class Encoder(nn.Module):
    """resnet34 encoder returning 5 skip feature maps"""

    def __init__(self, in_channels, pretrained=True):
        super().__init__()
        weight = ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = resnet34(weights=weight)

        # adapt conv1 for 2 or 5 channels
        if in_channels != 3:
            old_conv = backbone.conv1
            new_conv = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
            if pretrained:
                w = old_conv.weight
                mean_w = w.mean(dim=1, keepdim=True)  # rgb mean filter for lidar channels

                with torch.no_grad():
                    if in_channels > 3:
                        new_conv.weight[:, :3] = w
                        new_conv.weight[:, 3:] = mean_w.repeat(1, in_channels - 3, 1, 1)
                    if in_channels < 3:
                        new_conv.weight[:] = mean_w.repeat(1, in_channels, 1, 1)

            backbone.conv1 = new_conv

        # drop avgpool and fc
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x):
        """return 5 feature maps: channels 64 64 128 256 512, strides 2 4 8 16 32"""
        # f0 (B,  64, H/2,  W/2 )
        # f1 (B,  64, H/4,  W/4 )
        # f2 (B, 128, H/8,  W/8 )
        # f3 (B, 256, H/16, W/16)
        # f4 (B, 512, H/32, W/32)
        f0 = self.relu(self.bn1(self.conv1(x)))
        f1 = self.layer1(self.maxpool(f0))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [f0, f1, f2, f3, f4]

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [7]:
# @title src/models/decoder.py
import torch
import torch.nn as nn

 # (B, in,  h, w) -> (B, out, h, w)
class DoubleConv(nn.Module):
    """[Conv3x3 -> BN -> ReLU] x 2, same as A2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Decoder(nn.Module):
    "channels 64 64 128 256 512 from encoder from f0 f1 f2 f3 f4"
    # f0 (B,  64, H/2,  W/2 )
    # f1 (B,  64, H/4,  W/4 )
    # f2 (B, 128, H/8,  W/8 )
    # f3 (B, 256, H/16, W/16)
    # f4 (B, 512, H/32, W/32)

    def __init__(self, skip_channels=(64, 64, 128, 256, 512), num_out = 1):
        super().__init__()
        FINAL_LAYER = 32
        c0, c1, c2, c3, c4 = skip_channels

    #stage1 : f4 -> f3 : c4 -> c3
        self.up1  = nn.ConvTranspose2d(c4, c3, kernel_size=2,stride=2)    # (B, 512, H/32, W/32) -> (B, 256, H/16, W/16)
        self.conv_up1 = DoubleConv(c3 * 2, c3)    # (B, 512, H/16, W/16) -> (B, 256, H/16, W/16)

    #stage2 : f3 -> f2 : c3 -> c2
        self.up2  = nn.ConvTranspose2d(c3, c2, kernel_size=2,stride=2)    # (B, 256, H/16, W/16) -> (B, 128, H/8, W/8)
        self.conv_up2 = DoubleConv(c2 * 2, c2)    # (B, 256, H/8, W/8) -> (B, 128, H/8, W/8)

    #stage3 : f2 -> f1 : c2 -> c1
        self.up3  = nn.ConvTranspose2d(c2, c1, kernel_size=2,stride=2)    # (B, 128, H/8, W/8) -> (B, 64, H/4, W/4)
        self.conv_up3 = DoubleConv(c1 * 2, c1)    # (B, 128, H/4, W/4) -> (B, 64, H/4, W/4)

    #stage4 : f1 -> f0 : c1 -> c0
        self.up4  = nn.ConvTranspose2d(c1, c0, kernel_size=2,stride=2)    # (B, 64, H/4, W/4) -> (B, 64, H/2, W/2)
        self.conv_up4 = DoubleConv(c0 * 2, c0)    # (B, 128, H/2, W/2) -> (B, 64, H/2, W/2)

    #stage5 : f0 -> full size : c0 -> FINAL_LAYER
        self.up5  = nn.ConvTranspose2d(c0, FINAL_LAYER, kernel_size=2,stride=2)    # (B, 64, H/2, W/2) -> (B, 32, H, W)
        self.conv_up5 = DoubleConv(FINAL_LAYER, FINAL_LAYER)    # (B, 32, H, W) -> (B, 32, H, W)

    #head: FINAL_LAYER -> num_out
        self.head = nn.Conv2d(FINAL_LAYER, num_out, kernel_size=1)    # (B, 32, H, W) -> (B, num_out, H, W)

    def forward(self, feats):
        f0, f1, f2, f3, f4 = feats

    #stage1 f4 + f3
        x = self.up1(f4)    # (B, 256, H/16, W/16)
        x = torch.cat([x, f3], dim=1)    # (B, 512, H/16, W/16)
        x = self.conv_up1(x)    # (B, 256, H/16, W/16)

    #stage2 f4 + f3 + f2
        x = self.up2(x)    # (B, 128, H/8, W/8)
        x = torch.cat([x, f2], dim=1)    # (B, 256, H/8, W/8)
        x = self.conv_up2(x)    # (B, 128, H/8, W/8)

    #stage3 f4 + f3 + f2 + f1
        x = self.up3(x)    # (B, 64, H/4, W/4)
        x = torch.cat([x, f1], dim=1)    # (B, 128, H/4, W/4)
        x = self.conv_up3(x)    # (B, 64, H/4, W/4)

    #stage4 f4 + f3 + f2 + f1 + f0
        x = self.up4(x)    # (B, 64, H/2, W/2)
        x = torch.cat([x, f0], dim=1)    # (B, 128, H/2, W/2)
        x = self.conv_up4(x)    # (B, 64, H/2, W/2)

    #stage5 to final layer
        x = self.up5(x)    # (B, 32, H, W)
        x = self.conv_up5(x)    # (B, 32, H, W)

    #head
        return self.head(x)    # (B, num_out, H, W)

In [8]:
# @title src/models/m1_rgb.py
import torch
import torch.nn as nn

class M1RGB(nn.Module):
    def __init__(self, pretrained=True, skip_channels=(64, 64, 128, 256, 512), max_depth=80.0):
        super().__init__()
        self.max_depth = float(max_depth)
        self.encoder = Encoder(in_channels=3, pretrained=pretrained)  # depth/80 + mask
        self.decoder = Decoder(skip_channels=tuple(skip_channels), num_out=1)

    def forward(self, rgb, lidar):
        return {"d": torch.sigmoid(self.decoder(self.encoder(rgb))) * self.max_depth}

def build_m1(cfg):
    """build m1 model from yaml config"""
    pretrained = cfg["model"].get("pretrained", True)
    skip_channels = cfg["model"].get("skip_channels", (64, 64, 128, 256, 512))
    max_depth = cfg["data"].get("max_depth", 80.0)
    return M1RGB(pretrained, skip_channels, max_depth)

In [9]:
# @title src/models/m3_early.py
import torch
import torch.nn as nn
class M3Early(nn.Module):
    """model 3: early fusion, rgb + lidar stacked to 5-ch"""

    def __init__(self, pretrained=True, skip_channels=(64, 64, 128, 256, 512), max_depth=80.0):
        super().__init__()
        self.max_depth = float(max_depth)
        self.encoder = Encoder(in_channels=5, pretrained=pretrained)    # 3 rgb + 2 lidar (depth/80 + mask)
        self.decoder = Decoder(skip_channels=tuple(skip_channels), num_out=1)

    def forward(self, rgb, lidar):
        x = torch.cat([rgb, lidar], dim=1)    # (B, 3, H, W) + (B, 2, H, W) -> (B, 5, H, W)
        feats = self.encoder(x)    # (B, 5, H, W) -> [f0 ... f4]
        x = self.decoder(feats)    # (B, 1, H, W) raw
        d = torch.sigmoid(x) * self.max_depth    # (B, 1, H, W) metres 0..80
        return {"d": d}

def build_m3(cfg):
    """build m3 model from yaml config"""
    pretrained = cfg["model"].get("pretrained", True)
    skip_channels = cfg["model"].get("skip_channels", (64, 64, 128, 256, 512))
    max_depth = cfg["data"].get("max_depth", 80.0)
    return M3Early(pretrained, skip_channels, max_depth)

class M3Split(nn.Module):
    """model 3 split stem: own conv1+bn1 for rgb and lidar, added before layer1"""

    def __init__(self, pretrained=True, skip_channels=(64, 64, 128, 256, 512), max_depth=80.0):
        super().__init__()
        self.max_depth = float(max_depth)
        self.encoder = Encoder(in_channels=3, pretrained=pretrained)    # rgb stem + resnet body
        lid = Encoder(in_channels=2, pretrained=pretrained)             # same init as m2, keep only its stem
        self.encoder.lid_conv1 = lid.conv1    # put inside encoder -> optimizer trains it
        self.encoder.lid_bn1 = lid.bn1        # own BatchNorm -> lidar not drowned by rgb
        self.decoder = Decoder(skip_channels=tuple(skip_channels), num_out=1)

    def forward(self, rgb, lidar):
        e = self.encoder
        f0 = e.relu(e.bn1(e.conv1(rgb)) + e.lid_bn1(e.lid_conv1(lidar)))    # (B, 64, H/2, W/2) fuse here
        f1 = e.layer1(e.maxpool(f0))    # (B, 64, H/4, W/4)
        f2 = e.layer2(f1)    # (B, 128, H/8, W/8)
        f3 = e.layer3(f2)    # (B, 256, H/16, W/16)
        f4 = e.layer4(f3)    # (B, 512, H/32, W/32)
        x = self.decoder([f0, f1, f2, f3, f4])    # (B, 1, H, W) raw
        d = torch.sigmoid(x) * self.max_depth    # (B, 1, H, W) metres 0..80
        return {"d": d}

def build_m3_split(cfg):
    """build m3 split-stem model from yaml config"""
    pretrained = cfg["model"].get("pretrained", True)
    skip_channels = cfg["model"].get("skip_channels", (64, 64, 128, 256, 512))
    max_depth = cfg["data"].get("max_depth", 80.0)
    return M3Split(pretrained, skip_channels, max_depth)

In [13]:
# @title src/train.py
import csv
import random
from pathlib import Path

import numpy as np
import torch


# model registry: Colab runs Model 1 and Model 3 only
MODELS = {"m1_rgb": build_m1, "m3_early": build_m3, "m3_split": build_m3_split}
LOSSES = {"m1_rgb": m1_loss, "m3_early": m3_loss, "m3_split": m3_loss}
#checkpoint:
def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_rmse, bad_epochs):
    #TODO pls make it fit with ur local, im doing colab ver prob dont use it
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "best_rmse": best_rmse,
        "bad_epochs": bad_epochs,
    }, path)

def load_checkpoint(path, model, optimizer, scheduler, scaler, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    return ckpt["epoch"] + 1, ckpt["best_rmse"], ckpt["bad_epochs"]    # next epoch to run

#2 build
def build_model(cfg):
    return MODELS[cfg["model"]["name"]](cfg)

def build_loss(cfg):
    return LOSSES[cfg["model"]["name"]]

def build_optimizer(model,cfg):
    train = cfg["train"]
    lr  = train["lr"]

    # AdamW: encoder lr = 0.1 x decoder lr
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": lr * train["encoder_lr_scale"]},
        {"params": model.decoder.parameters(), "lr": lr},
    ], weight_decay=train["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train["epochs"])
    return optimizer, scheduler

def build_scaler(cfg, device):
    "With AMP, much of the model runs in float16 instead of float32. That's faster and uses less GPU memory but when scaler up  loss some detail so use  sacler"
    use_amp = cfg["train"]["amp"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    return scaler, use_amp

def build_data(cfg, split):
    data_dir = Path(cfg["data"]["data_dir"])
    return KittiDepthDataset(data_dir / split, data_dir, split, Transform(), batch_size=cfg["data"]["batch_size"], seed=cfg["seed"], cache=cfg["data"]["cache"])

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

#3  one  epoch
def train_one_epoch(model, data, loss_fn, optimizer, scaler, device, use_amp, max_batches=0):
    model.train()
    total_loss, n = 0.0, 0
    n_batches = (len(data) + data.batch_size - 1) // data.batch_size
    for i, batch in enumerate(data.get_batch()):
        if max_batches and i >= max_batches:  break
        rgb = batch["rgb"].to(device)
        lidar = batch["lidar"].to(device)
        gt =  batch["gt"].to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out  = model(rgb, lidar)
            loss = loss_fn(out["d"], gt)
        scaler.scale(loss).backward()    # loss x big number -> backward
        scaler.step(optimizer)    # grads / big number -> optimizer step
        scaler.update()    # adjust the big number for next batch

        total_loss += loss.item()
        n +=  1
        if n % 50 == 0:    # progress every 50 batches
            print(f"  batch {n}/{n_batches} | avg loss {total_loss/n:.4f}")
    return total_loss/max(n,1)

@torch.no_grad()
def validate(model, data, device, use_amp, max_depth, max_batches=0):
    model.eval()
    metric = Metric(max_depth=max_depth, by_range=True)    # also RMSE/MAE per depth range
    for i, batch in enumerate(data.get_batch()):
        if max_batches and i >= max_batches: break
        rgb = batch["rgb"].to(device)    # (B, 3, H, W)
        lidar = batch["lidar"].to(device)    # (B, 2, H, W)
        gt = batch["gt"].to(device)    # (B, 1, H, W)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(rgb, lidar)
        metric.update(out["d"], gt)
    return metric.compute()



def train_n_epochs(model, train_data, val_data, loss_fn, optimizer, scheduler, scaler, device, use_amp, cfg, run_dir, max_batches=0):
    train = cfg["train"]
    max_depth = cfg["data"]["max_depth"]
    last_path, best_path, log_path = run_dir / "last.pt", run_dir / "best.pt", run_dir / "log.csv"

    # resume if last.pt exists
    start_epoch, best_rmse, bad_epochs = 0, float("inf"), 0
    if last_path.exists():
        start_epoch, best_rmse, bad_epochs = load_checkpoint(last_path, model, optimizer, scheduler, scaler, device)
        print(f"resume from epoch {start_epoch}")

    for epoch in range(start_epoch, train["epochs"]):
        if device.type == "cuda": torch.cuda.reset_peak_memory_stats(device)
        lr = optimizer.param_groups[1]["lr"]    # decoder lr used this epoch

        with TrainingTimer(device) as timer:
            train_loss = train_one_epoch(model, train_data, loss_fn, optimizer, scaler, device, use_amp, max_batches)
            val = validate(model, val_data, device, use_amp, max_depth, max_batches)
        scheduler.step()
        peak_mem = torch.cuda.max_memory_allocated(device) / 1024**3 if device.type == "cuda" else 0.0    # GB

        # log.csv
        new_file = not log_path.exists()
        with open(log_path, "a", newline="") as f:
            w = csv.writer(f)
            ranges = [k for k, v in val.items() if isinstance(v, dict)]    # "0-20", "20-40", "40-80.0"
            if new_file: w.writerow(["epoch", "train_loss", "val_rmse", "val_mae", "val_valid_ratio"]
                                    + [f"{m}_{r}" for r in ranges for m in ("rmse", "mae")]
                                    + ["lr", "time", "peak_mem"])
            w.writerow([epoch, train_loss, val["rmse"], val["mae"], val["%valid/total_pixel"]]
                       + [val[r][m] for r in ranges for m in ("rmse", "mae")]
                       + [lr, timer.time, peak_mem])
        print(f"epoch {epoch} | loss {train_loss:.4f} | val_rmse {val['rmse']:.4f} | val_mae {val['mae']:.4f} | {timer.time:.1f}s | {peak_mem:.2f}GB")

        # best.pt by val rmse + early stopping counter
        if val["rmse"] < best_rmse:
            best_rmse, bad_epochs = val["rmse"], 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_rmse": best_rmse}, best_path)
        else:
            bad_epochs += 1

        save_checkpoint(last_path, model, optimizer, scheduler, scaler, epoch, best_rmse, bad_epochs)

        if bad_epochs >= train["early_stopping_patience"]:
            print(f"early stop: no better val rmse for {bad_epochs} epochs")
            break

In [ ]:
# @title 4. Config (base.yaml as a dict)
cfg = {
    "seed": 666,
    "data": {
        "data_dir": DATA_DIR,
        "batch_size": BATCH_SIZE,
        "max_depth": 80.0,
        "cache": CACHE_IN_RAM,                     # keep transformed samples in RAM
    },
    "model": {
        "name": MODEL,                             # m1_rgb | m3_early
        "pretrained": True,                        # ResNet34_Weights.IMAGENET1K_V1
        "skip_channels": [64, 64, 128, 256, 512],
    },
    "train": {
        "epochs": EPOCHS,
        "lr": 0.0002,                              # decoder lr
        "encoder_lr_scale": 0.1,                   # encoder lr = 0.1 * lr
        "weight_decay": 0.0001,
        "amp": True,
        "early_stopping_patience": 5,
        "max_batches": MAX_BATCHES,
    },
    "output": {
        "checkpoints_dir": DRIVE_RUNS,             # Google Drive
    },
}

In [ ]:
# @title 5. Setup: fresh start or resume from Drive
import time, yaml
from pathlib import Path

RUN_NAMES = {"m1_rgb": "m1", "m3_early": "m3", "m3_split": "m3_split"}
name = RUN_NAMES[MODEL]                                          # folder under checkpoints/
run_dir = Path(cfg["output"]["checkpoints_dir"]) / name          # MyDrive/KITTI_DC/checkpoints/m1

if RESUME:
    assert (run_dir / "last.pt").exists(), f"nothing to resume: {run_dir}/last.pt not found"
    print("resume from", run_dir / "last.pt")                    # train_n_epochs loads it
else:
    if run_dir.exists():
        old = f"{run_dir}_old_{time.strftime('%m%d-%H%M')}"
        os.rename(run_dir, old)                                  # keep the old run, do not resume it
        print("old run moved to", old)
    print("fresh start")
run_dir.mkdir(parents=True, exist_ok=True)

with open(run_dir / "config.yaml", "w") as f:
    yaml.safe_dump(cfg, f)
set_seed(cfg["seed"])
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True    # fixed 352x1216 input -> cuDNN picks the fastest conv kernels
max_batches = cfg["train"]["max_batches"]
print("run dir:", run_dir)

old run moved to /content/drive/MyDrive/KITTI_DC/checkpoints/m3_split_old_1128-1632
fresh start
run dir: /content/drive/MyDrive/KITTI_DC/checkpoints/m3_split


In [ ]:
# @title 6. Train
train_data = build_data(cfg, "train")    # data_dir/train/, shuffled
val_data = build_data(cfg, "val")    # data_dir/val/, different drives
model = build_model(cfg).to(device)
loss_fn = build_loss(cfg)
optimizer, scheduler = build_optimizer(model, cfg)
scaler, use_amp = build_scaler(cfg, device)

print(f"run {name} | device {device} | amp {use_amp} | train {len(train_data)} | val {len(val_data)}")
train_n_epochs(model, train_data, val_data, loss_fn, optimizer, scheduler, scaler, device, use_amp, cfg, run_dir, max_batches)

  cached train: 1000/8237
  cached train: 2000/8237
  cached train: 3000/8237
  cached train: 4000/8237
  cached train: 5000/8237
  cached train: 6000/8237
  cached train: 7000/8237
  cached train: 8000/8237
  cached train: 8237/8237
  cached val: 1000/1023
  cached val: 1023/1023
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 179MB/s]


run m3_split | device cuda | amp True | train 8237 | val 1023
  batch 50/1030 | avg loss 17.3176
  batch 100/1030 | avg loss 13.0889
  batch 150/1030 | avg loss 10.9804
  batch 200/1030 | avg loss 9.6350
  batch 250/1030 | avg loss 8.6068
  batch 300/1030 | avg loss 7.8019
  batch 350/1030 | avg loss 7.1906
  batch 400/1030 | avg loss 6.6573
  batch 450/1030 | avg loss 6.2209
  batch 500/1030 | avg loss 5.8499
  batch 550/1030 | avg loss 5.5376
  batch 600/1030 | avg loss 5.2638
  batch 650/1030 | avg loss 5.0302
  batch 700/1030 | avg loss 4.8185
  batch 750/1030 | avg loss 4.6222
  batch 800/1030 | avg loss 4.4466
  batch 850/1030 | avg loss 4.2932
  batch 900/1030 | avg loss 4.1534
  batch 950/1030 | avg loss 4.0197
  batch 1000/1030 | avg loss 3.9027
epoch 0 | loss 3.8353 | val_rmse 1.5279 | val_mae 0.6932 | 155.8s | 7.00GB
  batch 50/1030 | avg loss 1.6023
  batch 100/1030 | avg loss 1.5844
  batch 150/1030 | avg loss 1.5906
  batch 200/1030 | avg loss 1.5693
  batch 250/1030 | av

In [ ]:
# @title 7. Show the log
import pandas as pd

pd.read_csv(run_dir / "log.csv")

,epoch,train_loss,val_rmse,val_mae,val_valid_ratio,rmse_0-20,mae_0-20,rmse_20-40,mae_20-40,rmse_40-80.0,mae_40-80.0,lr,time,peak_mem
0,0,3.835329,1.527883,0.693241,0.140064,0.817484,0.450749,2.362442,1.366820,4.418095,2.759733,0.000200,155.817028,7.003785
1,1,1.417391,1.221828,0.531469,0.140064,0.638500,0.352952,1.749847,0.972943,3.713102,2.166598,0.000199,123.676759,3.935329
2,2,1.215661,1.117668,0.454521,0.140064,0.562578,0.292026,1.640799,0.829161,3.409135,2.003714,0.000195,123.816799,3.935329
3,3,1.104691,1.069505,0.446208,0.140064,0.579637,0.308082,1.531243,0.754875,3.194648,1.782411,0.000189,123.954533,3.935329
4,4,1.040636,1.039635,0.410647,0.140064,0.484443,0.268198,1.533481,0.730229,3.251502,1.787458,0.000181,124.212306,3.935329
5,5,0.995934,0.992502,0.379192,0.140064,0.476644,0.241255,1.513334,0.724239,3.020922,1.634593,0.000171,123.845691,3.935329
6,6,0.935719,0.990927,0.376959,0.140064,0.458126,0.231791,1.439011,0.723468,3.130814,1.735457,0.000159,124.176947,3.935329
7,7,0.904969,0.990417,0.396992,0.140064,0.484450,0.262339,1.409847,0.644646,3.105925,1.821825,0.000145,123.938545,3.935329
8,8,0.874629,0.972955,0.355891,0.140064,0.434440,0.211222,1.330840,0.666486,3.178463,1.784507,0.000131,123.848804,3.935329
9,9,0.839967,0.957755,0.327588,0.140064,0.410136,0.197167,1.386347,0.626958,3.100432,1.575130,0.000116,124.166143,3.935329


In [16]:
# @title 7b. Test best.pt of a chosen run (reuses GPU + data on /content/kitti)
TEST_RUN = "m1_rmse"    # @param ["m3_split_batch_8", "m3_split_batch_40", "m1_rmse"] {allow-input: true}
import gc, json, yaml

DRIVE_RESULTS = "/content/drive/MyDrive/KITTI_DC/results"
if "train_data" in globals(): del train_data; gc.collect()    # free the train cache in RAM

test_dir = Path(DRIVE_RUNS) / TEST_RUN
assert (test_dir / "best.pt").exists(), f"not found: {test_dir}/best.pt"
test_cfg = yaml.safe_load((test_dir / "config.yaml").read_text())
test_cfg["model"]["pretrained"] = False                  # weights come from best.pt
test_cfg["data"]["data_dir"] = DATA_DIR
test_cfg["data"]["cache"] = False                        # read once, no need to cache

device = torch.device("cuda")
use_amp = test_cfg["train"]["amp"]
ckpt = torch.load(test_dir / "best.pt", map_location=device, weights_only=False)
model = build_model(test_cfg).to(device)                 # m1_rgb / m3_early / m3_split from config
model.load_state_dict(ckpt["model"])

test_data = build_data(test_cfg, "test")
with TrainingTimer(device) as timer:
    res = validate(model, test_data, device, use_amp, test_cfg["data"]["max_depth"])

n = lambda d: d.get("total_valid ", d.get("total_valid", 0))    # Metric key has a trailing space
out = {
    "run_id": TEST_RUN,
    "model": test_cfg["model"]["name"],
    "split": "test",
    "best_epoch": ckpt["epoch"],
    "val_rmse": ckpt["val_rmse"],
    "rmse": res["rmse"],
    "mae": res["mae"],
    "n_valid": n(res),
    "n_images": len(test_data),
    "time_s": timer.time,
    "ms_per_img": timer.time * 1000.0 / len(test_data),
    "fps": len(test_data) / timer.time,
    "gpu": timer.device_name,
    "batch_size": test_cfg["data"]["batch_size"],
    "per_range": {k: {"rmse": v["rmse"], "mae": v["mae"], "n_valid": n(v)} for k, v in res.items() if isinstance(v, dict)},
}

out_dir = Path(DRIVE_RESULTS) / TEST_RUN
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "metrics_test.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))

{
  "run_id": "m1_rmse",
  "model": "m1_rgb",
  "split": "test",
  "best_epoch": 15,
  "val_rmse": 2.8328560465755572,
  "rmse": 4.2356822767129,
  "mae": 1.954380585542348,
  "n_valid": 73128306,
  "n_images": 1000,
  "time_s": 47.29234293500008,
  "ms_per_img": 47.29234293500008,
  "fps": 21.145072075926286,
  "gpu": "NVIDIA A100-SXM4-80GB",
  "batch_size": 40,
  "per_range": {
    "0-20": {
      "rmse": 2.3200513166104004,
      "mae": 1.0420600854811481,
      "n_valid": 57604748
    },
    "20-40": {
      "rmse": 5.428870694330048,
      "mae": 3.8258555199496658,
      "n_valid": 11564254
    },
    "40-80.0": {
      "rmse": 12.863078297503213,
      "mae": 9.713682192148987,
      "n_valid": 3940307
    }
  }
}


In [17]:
# @title 8. Close: finish the upload to Drive + shut down the runtime
from google.colab import runtime

# show what is on Drive before closing
for folder in (DRIVE_RUNS, "/content/drive/MyDrive/KITTI_DC/results"):
    print(folder)
    for run in sorted(Path(folder).glob("*")):
        print("  ", run.name, [f.name for f in run.iterdir()])

drive.flush_and_unmount()    # wait until every file (best.pt, last.pt, log.csv, metrics_test.json) is uploaded
print("all saved on Drive")
runtime.unassign()           # end the session (stops using compute units)

/content/drive/MyDrive/KITTI_DC/checkpoints
   m1_rmse ['config.yaml', 'best.pt', 'log.csv', 'last.pt']
   m3_split_batch_40 ['config.yaml', 'best.pt', 'log.csv', 'last.pt']
   m3_split_batch_8 ['config.yaml', 'best.pt', 'log.csv', 'last.pt']
/content/drive/MyDrive/KITTI_DC/results
   m1_rmse ['metrics_test.json']
   m3_split_batch_40 ['metrics_test.json']
   m3_split_batch_8 ['metrics_test.json']
all saved on Drive
